In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("data")

# ---------- 0) Loader con fallback di separatore ----------
def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

# ---------- 1) Master list comuni (fonte di verità per etichette) ----------
# municipalities_trentino.csv ha: id, comune, latitude, longitude
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
master_ids   = municipal["id"].tolist()
name_by_id   = dict(zip(municipal["id"], municipal["comune_norm"]))
id_by_name   = dict(zip(municipal["comune_norm"], municipal["id"]))

# ---------- 2) Utilities su tempo e allineamento ----------
def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s:
        return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

def minutes_to_hmm(minutes):
    if pd.isna(minutes):
        return None
    minutes = int(minutes)
    h, m = divmod(minutes, 60)
    return f"{h}:{m:02d}"

def ensure_matrix_labeled_and_ordered(df, master_names, index_col_name="Unnamed: 0"):
    """
    - Usa la prima colonna come indice (nomi dei comuni)
    - Verifica stessa cardinalità/insieme di etichette su righe/colonne
    - Riordina esattamente come master_names
    - Ritorna (df_riordinato, report_test)
    """
    # Imposta indice = colonna con i nomi riga
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    # Rimuovi spazi
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()

    rows = set(df.index)
    cols = set(df.columns)
    master = set(master_names)

    errs = []
    if rows != master:
        miss_r = sorted(list(master - rows))
        extra_r = sorted(list(rows - master))
        errs.append(f"[RIGHE] Mancano nel matrix: {miss_r[:10]} ...; Extra: {extra_r[:10]} ...")
    if cols != master:
        miss_c = sorted(list(master - cols))
        extra_c = sorted(list(cols - master))
        errs.append(f"[COLONNE] Mancano nel matrix: {miss_c[:10]} ...; Extra: {extra_c[:10]} ...")

    if errs:
        raise ValueError("Etichette non coerenti con master list:\n" + "\n".join(errs))

    # Riordina
    df = df.loc[master_names, master_names]
    return df

# ---------- 3) Allinea le matrici distanza/tempo ai nomi master ----------
dist_km_aligned = ensure_matrix_labeled_and_ordered(dist_km.copy(), master_names)
time_hm_aligned = ensure_matrix_labeled_and_ordered(time_hm.copy(), master_names)

# Converte time hh:mm in minuti (stessa etichettatura/ordine)
time_min_aligned = time_hm_aligned.applymap(hmm_to_minutes)

# ---------- 4) Normalizza incoming/outcoming/pop con Codice+Comune ----------
# incoming/outcoming hanno: Codice, Comune, Stesso, Altro, Totale, Lordo, Netto
# pop ha: Codice, popolazione
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)

pop["Codice"] = pop["Codice"].astype(int)

# Controlli di coerenza:
assert incoming["Comune_norm"].isin(municipal["comune_norm"]).all(), "Nomi incoming non allineati alla master list"
assert outcoming["Comune_norm"].isin(municipal["comune_norm"]).all(), "Nomi outcoming non allineati alla master list"
assert set(incoming["Codice"]) == set(municipal["id"]), "Codici incoming diversi da id master"
assert set(outcoming["Codice"]) == set(municipal["id"]), "Codici outcoming diversi da id master"
assert set(pop["Codice"]) == set(municipal["id"]), "Codici pop diversi da id master"

# Join rapido di popolazione per nome e codice (ridondante ma sicuro)
incoming = (incoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

outcoming = (outcoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

# ---------- 5) Calcolo indicatori base (uscite) ----------
# Denominatore = persone attive residenti (stesso + netto)
den_out = (outcoming["Stesso"] + outcoming["Netto"]).replace({0: np.nan})
out_inds = outcoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
out_inds["quota_stesso"] = out_inds["Stesso"] / den_out
out_inds["quota_intra"]  = out_inds["Altro"]  / den_out
out_inds["quota_extra"]  = (out_inds["Netto"] - out_inds["Altro"]) / den_out

# ---------- 6) Indicatori in entrata e saldo ----------
in_inds = incoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
in_inds = in_inds.rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})

# Merge entrata/uscita per comune
indicators = (out_inds
              .merge(in_inds[["Codice","in_Stesso","in_Altro","in_Lordo","in_Netto","in_Totale"]],
                     on="Codice", how="left"))

# Saldi e intensità
indicators["saldo_netto"]   = indicators["in_Totale"] - indicators["Totale"]
indicators["indice_bilancio"] = np.where(indicators["Totale"]>0,
                                         indicators["in_Totale"]/indicators["Totale"], np.nan)
indicators["intensita_pendolarismo"] = (indicators["in_Totale"] + indicators["Totale"]) / indicators["popolazione"]
indicators["autocontenimento_pc"] = indicators["Stesso"] / indicators["popolazione"]
indicators["attrattivita_pc"]     = indicators["in_Totale"] / indicators["popolazione"]

# ---------- 7) Controlli finali su coerenza etichette ----------
def assert_same_order_names(df_matrix, names):
    assert list(df_matrix.index) == names, "Index matrix non coincide con master_names"
    assert list(df_matrix.columns) == names, "Columns matrix non coincide con master_names"

assert_same_order_names(dist_km_aligned, master_names)
assert_same_order_names(time_hm_aligned, master_names)

# ---------- 8) Export "puliti" pronti per le analisi successive ----------
EXPORT = DATA  # cambia se vuoi una cartella dedicata
municipal[["id","comune","latitude","longitude"]].to_csv(EXPORT/"comuni_master.csv", index=False)
dist_km_aligned.to_csv(EXPORT/"distance_km_matrix_aligned.csv")
time_hm_aligned.to_csv(EXPORT/"time_hhmm_matrix_aligned.csv")
time_min_aligned.to_csv(EXPORT/"time_minutes_matrix_aligned.csv")
indicators.to_csv(EXPORT/"indicators_by_comune_2021.csv", index=False)

print("OK ✅  Etichette lette dai file & allineate.\n"
      f"- comuni_master.csv: {len(municipal)} comuni\n"
      f"- distance_km_matrix_aligned.csv: {dist_km_aligned.shape}\n"
      f"- time_minutes_matrix_aligned.csv: {time_min_aligned.shape}\n"
      f"- indicators_by_comune_2021.csv: {indicators.shape}")


OK ✅  Etichette lette dai file & allineate.
- comuni_master.csv: 166 comuni
- distance_km_matrix_aligned.csv: (166, 166)
- time_minutes_matrix_aligned.csv: (166, 166)
- indicators_by_comune_2021.csv: (166, 22)


/tmp/ipykernel_31382/412498493.py:85: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min_aligned = time_hm_aligned.applymap(hmm_to_minutes)


In [11]:
# Phase 1 — Validation & Rankings report for Trentino commuter data (2021)
# This cell builds on the project files already present in /mnt/data.
# It produces CSV exports and a few quick charts.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("data")
EXPORT = DATA / "exports_phase1"
EXPORT.mkdir(exist_ok=True)

def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

# Load base data
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

# --- Prepare master names ---
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
id_by_name   = dict(zip(municipal["comune_norm"], municipal["id"]))

# Normalize incoming/outcoming/pop
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)
pop["Codice"] = pop["Codice"].astype(int)

# Merge population
incoming = (incoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

outcoming = (outcoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

# Derived indicators
den_out = (outcoming["Stesso"] + outcoming["Netto"]).replace({0: np.nan})

out_inds = outcoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
out_inds["quota_stesso"] = out_inds["Stesso"] / den_out
out_inds["quota_intra"]  = out_inds["Altro"]  / den_out
out_inds["quota_extra"]  = (out_inds["Netto"] - out_inds["Altro"]) / den_out

in_inds = incoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
in_inds = in_inds.rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})

indicators = (out_inds
              .merge(in_inds[["Codice","in_Stesso","in_Altro","in_Lordo","in_Netto","in_Totale"]],
                     on="Codice", how="left"))

indicators["saldo_netto"]   = indicators["in_Totale"] - indicators["Totale"]
indicators["indice_bilancio"] = np.where(indicators["Totale"]>0,
                                         indicators["in_Totale"]/indicators["Totale"], np.nan)
indicators["intensita_pendolarismo"] = (indicators["in_Totale"] + indicators["Totale"]) / indicators["popolazione"]
indicators["autocontenimento_pc"] = indicators["Stesso"] / indicators["popolazione"]
indicators["attrattivita_pc"]     = indicators["in_Totale"] / indicators["popolazione"]

# --- Provincial consistency check (incoming vs outcoming totals) ---
prov_check = pd.DataFrame({
    "tot_entrate": [int(incoming["Totale"].sum())],
    "tot_uscite":  [int(outcoming["Totale"].sum())],
    "diff":        [int(incoming["Totale"].sum() - outcoming["Totale"].sum())]
})
prov_check.to_csv(EXPORT/"provincial_consistency_check.csv", index=False)

# --- Rankings (top/bottom 10) ---
def top_bottom(df, metric, n=10):
    base = df[["Codice","Comune_norm",metric]].dropna().copy()
    top = base.sort_values(metric, ascending=False).head(n).reset_index(drop=True)
    bottom = base.sort_values(metric, ascending=True).head(n).reset_index(drop=True)
    top["rank"] = np.arange(1, len(top)+1)
    bottom["rank"] = np.arange(1, len(bottom)+1)
    return top, bottom

rank_metrics = ["saldo_netto", "intensita_pendolarismo", "autocontenimento_pc", "attrattivita_pc"]
rank_exports = {}

for m in rank_metrics:
    top, bottom = top_bottom(indicators, m, n=10)
    top.to_csv(EXPORT/f"ranking_top10_{m}.csv", index=False)
    bottom.to_csv(EXPORT/f"ranking_bottom10_{m}.csv", index=False)
    rank_exports[m] = (top, bottom)

# Combine a single wide ranking table (with zscore for comparability)
from scipy.stats import zscore

rank_wide = indicators[["Codice","Comune_norm","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]].copy()
for c in ["saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]:
    rank_wide[c+"_z"] = zscore(rank_wide[c].fillna(rank_wide[c].mean()))

rank_wide["composite_score"] = rank_wide[[c+"_z" for c in ["saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]]].mean(axis=1)
rank_wide = rank_wide.sort_values("composite_score", ascending=False)
rank_wide.to_csv(EXPORT/"ranking_composite.csv", index=False)

In [14]:

# Display to the user
print("Validation — provincial totals (entrate vs uscite)")
prov_check


Validation — provincial totals (entrate vs uscite)


,tot_entrate,tot_uscite,diff
0,4948,7343,-2395


In [ ]:
print("Ranking (composite score, top)") 
rank_wide.head(20)

Ranking (composite score, top)


,Codice,Comune_norm,saldo_netto,intensita_pendolarismo,autocontenimento_pc,attrattivita_pc,saldo_netto_z,intensita_pendolarismo_z,autocontenimento_pc_z,attrattivita_pc_z,composite_score
16,238,Borgo Chiese,167,0.116896,0.148545,0.101072,3.405064,3.747815,0.837130,7.395376,3.846346
139,183,Storo,151,0.096931,0.178847,0.065136,3.104773,2.961450,1.455330,4.565085,3.021660
68,95,Grigno,115,0.099611,0.156463,0.077745,2.429120,3.067020,0.998654,5.558202,3.013249
118,161,Rovereto,152,0.020020,0.227321,0.011917,3.123541,-0.067778,2.444237,0.373614,1.468404
86,118,Moena / Moena,71,0.037557,0.169575,0.032246,1.603321,0.622932,1.266164,1.974711,1.366782
8,7,Avio,23,0.057946,0.161614,0.031785,0.702449,1.425990,1.103742,1.938404,1.292646
121,164,Sagron Mis,-29,0.184358,0.039106,0.011173,-0.273495,6.404857,-1.395549,0.315055,1.262717
13,17,Bleggio Superiore,56,0.045902,0.099016,0.041311,1.321798,0.951600,-0.173313,2.688711,1.197199
71,102,Lavarone,25,0.037911,0.181129,0.029486,0.739985,0.636866,1.501874,1.757358,1.159021
19,22,Borgo Valsugana,88,0.027217,0.175333,0.019911,1.922379,0.215671,1.383632,1.003250,1.131233


In [ ]:
print("Ranking (composite score, bottom)")
rank_wide.tail(20)


Ranking (composite score, bottom)


,Codice,Comune_norm,saldo_netto,intensita_pendolarismo,autocontenimento_pc,attrattivita_pc,saldo_netto_z,intensita_pendolarismo_z,autocontenimento_pc_z,attrattivita_pc_z,composite_score
90,127,Nogaredo,-13,0.009201,0.062954,0.001453,0.026796,-0.493902,-0.909026,-0.450512,-0.456661
43,54,Cavizzana,-1,0.004167,0.066667,0.000000,0.252013,-0.692184,-0.833284,-0.564932,-0.459597
64,90,Frassilongo / Garait,0,0.000000,0.073529,0.000000,0.270782,-0.856294,-0.693276,-0.564932,-0.460930
58,79,Dro,-38,0.011499,0.078311,0.001983,-0.442408,-0.403399,-0.595729,-0.408788,-0.462581
21,26,Bresimo,-3,0.012000,0.052000,0.000000,0.214477,-0.383659,-1.132500,-0.564932,-0.466653
151,202,Torcegno,-3,0.004418,0.064801,0.000000,0.214477,-0.682275,-0.871342,-0.564932,-0.476018
10,11,Bedollo,-9,0.007478,0.059143,0.000680,0.101868,-0.561767,-0.986766,-0.511391,-0.489514
48,61,Civezzano,-22,0.008321,0.065590,0.001468,-0.142118,-0.528557,-0.855252,-0.449280,-0.493802
161,222,Villa Lagarina,-21,0.009084,0.059694,0.001817,-0.123350,-0.498516,-0.975539,-0.421845,-0.504812
65,91,Garniga Terme,-3,0.007653,0.051020,0.000000,0.214477,-0.554868,-1.152485,-0.564932,-0.514452


In [18]:
# --- Quick charts (matplotlib, one plot each, default colors) ---
def save_bar_top(df, metric, title, fname):
    top = df.sort_values(metric, ascending=False).head(10)
    plt.figure(figsize=(10,4))
    plt.bar(top["Comune_norm"], top[metric])
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(EXPORT/fname)
    plt.close()

save_bar_top(indicators, "saldo_netto", "Top 10 comuni per saldo netto (entrate - uscite)", "chart_top10_saldo_netto.png")
save_bar_top(indicators, "intensita_pendolarismo", "Top 10 intensità di pendolarismo (entrate+uscite / popolazione)", "chart_top10_intensita.png")
save_bar_top(indicators, "attrattivita_pc", "Top 10 attrattività pro capite (entrate/pop)", "chart_top10_attrattivita.png")
save_bar_top(indicators, "autocontenimento_pc", "Top 10 autocontenimento pro capite (stesso/pop)", "chart_top10_autocontenimento.png")

# Export a clean indicators table
indicators_sorted = indicators.sort_values("Comune_norm")
indicators_sorted.to_csv(EXPORT/"indicators_by_comune_2021_phase1.csv", index=False)

# Final message to the notebook output
print("Phase 1 complete.\n"
      f"Exports in: {EXPORT}\n"
      "- provincial_consistency_check.csv\n"
      "- ranking_top10_*.csv / ranking_bottom10_*.csv\n"
      "- ranking_composite.csv\n"
      "- indicators_by_comune_2021_phase1.csv\n"
      "- chart_*.png")


Phase 1 complete.
Exports in: data/exports_phase1
- provincial_consistency_check.csv
- ranking_top10_*.csv / ranking_bottom10_*.csv
- ranking_composite.csv
- indicators_by_comune_2021_phase1.csv
- chart_*.png
